# Module 01 — Agent Skills

**Released:** v1.12.0 (March 24, 2026)

Skills are reusable instruction packs with progressive disclosure (metadata → instructions → resources). An agent loads skills from disk; the framework injects them into the system prompt at the right time.

We'll cover:
1. Discover skills on disk (metadata level)
2. Activate a skill (loads the full body)
3. Format a skill for prompt injection
4. Attach skills to an `Agent`
5. Run the full demo via a `Flow`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from pathlib import Path
SKILLS_DIR = Path.cwd().parent / "skills"
print("Skills dir:", SKILLS_DIR)
print("Contents:", [p.name for p in SKILLS_DIR.iterdir() if p.is_dir()])

## 1. Discover skills

`discover_skills()` scans a directory for `SKILL.md` files and returns `Skill` objects at the **METADATA** disclosure level — frontmatter only, no body. That keeps startup cheap when an agent has many skills.

In [ ]:
from crewai.skills import discover_skills, activate_skill
from crewai.skills.loader import format_skill_context

skills = discover_skills(SKILLS_DIR)
for s in skills:
    print(f"- {s.name} (disclosure_level={s.disclosure_level}): {s.frontmatter.description}")

## 2. Activate a skill

`activate_skill()` promotes a skill to **INSTRUCTIONS** level — the full Markdown body is loaded and ready for prompt injection.

In [ ]:
researcher = next(s for s in skills if s.name == "researcher-skill")
activated = activate_skill(researcher)
print("disclosure_level:", activated.disclosure_level)
print("---\n")
print(format_skill_context(activated)[:500], "\n...")

## 3. Attach skills to an Agent

The ergonomic path: pass skill directories directly as `skills=[...]`. CrewAI discovers and activates them automatically during agent setup.

In [ ]:
from crewai import Agent
from showcase.shared import get_llm

# IMPORTANT: pass the PARENT dir (SKILLS_DIR), not individual skill folders.
# `discover_skills()` scans for SKILL.md files inside subdirectories of the
# path it's given — pointing it at a specific skill dir yields zero hits.
agent = Agent(
    role="Research Analyst",
    goal="Produce a well-cited brief on multi-agent coordination",
    backstory="Senior analyst who follows installed skills to the letter.",
    llm=get_llm(),
    skills=[SKILLS_DIR],
)
print(f"Agent loaded {len(agent.skills or [])} skill(s): "
      f"{[s.name for s in agent.skills or []]}")

## 4. Run the Flow

`SkillsFlow` is a one-step Flow that kicks off the research task. Running inside a Flow gives you state + observability you can extend to multi-step pipelines later.

In [ ]:
from showcase.flows.skills_flow import SkillsFlow

flow = SkillsFlow()
flow.kickoff(inputs={"topic": "The 2026 state of open-source agent frameworks"})
print(flow.state.result)

## Try this

- Add a new skill under `skills/your-skill/SKILL.md` and watch it appear in `discover_skills()`.
- Filter skills by `frontmatter.allowed_tools` to gate behavior.
- Remove the `citation-skill` from the agent and compare output quality.